## First Time Period Aggregation

In [60]:
import pandas as pd
import numpy as np

df = pd.read_csv("post_chelsea_2017.csv")
acs = pd.read_csv("acs_all_2013_2023.csv")

In [61]:
# only grab first time period from 2013-2018
acs = acs.head(8)

In [62]:
# 'census_tract' is the column name in both dataframes
merged_df = pd.merge(df, acs, 
                    on='census_tract',  
                    how='left')  # Left merge keeps all rows from df

columns_to_average = [col for col in acs.columns if col not in ['census_tract', 'geoid', 'start_date', 'end_date']]

# Select specific columns to average'
grouped_df = merged_df.groupby(['ward_number', 'precinct_number'])[columns_to_average].mean()

# Round all numeric columns to 2 decimal places
grouped_df = grouped_df.round(2)

# Merge back with the original df
final_df = pd.merge(df, grouped_df, 
                   on=['ward_number', 'precinct_number'],
                   how='left')

In [63]:
election_columns = [col for col in final_df.columns if col not in ['voter_id_number', 'last_name', 'first_name', 'ward_number', 'precinct_number', 'longitude', 'latitude', 'census_tract', 'date_of_birth', 'date_of_registration'] + list(columns_to_average)]

melted_df1 = pd.melt(final_df,
                   id_vars=['voter_id_number', 'last_name', 'first_name', 'ward_number', 'precinct_number', 'longitude', 'latitude', 'census_tract', 'date_of_birth', 'date_of_registration'] + list(columns_to_average),  
                   value_vars=election_columns,  # These are your election date columns
                   var_name='election_date',  # This will contain the election dates
                   value_name='voted')  

# Sort the results for better readability
melted_df1 = melted_df1.sort_values(['voter_id_number', 'election_date'])

In [64]:
# Fill in blanks in voted column with 0
melted_df1['voted'] = melted_df1['voted'].fillna(0)

# Convert election_date and date_of_registration to datetime if they aren't already
melted_df1['election_date'] = pd.to_datetime(melted_df1['election_date'])
melted_df1['date_of_registration'] = pd.to_datetime(melted_df1['date_of_registration'])

# Create eligible column: 1 if registered at least 10 days before election, 0 otherwise
melted_df1['eligible'] = ((melted_df1['election_date'] - melted_df1['date_of_registration']).dt.days >= 10).astype(int)

# Sort the results for better readability
melted_df1 = melted_df1.sort_values(['voter_id_number', 'election_date'])

## Second Time Period Aggregation

In [67]:
df2 = pd.read_csv("post_chelsea_2022.csv")
acs2 = pd.read_csv("acs_all_2013_2023.csv")

In [69]:
# grab only the second time period from 2018-2023
acs2 = acs2.tail(9)

# merge the two dataframes
merged_df2 = pd.merge(df2, acs2, 
                    on='census_tract',  
                    how='left')  # Left merge keeps all rows from df

# Select specific columns to average        
columns_to_average = [col for col in acs2.columns if col not in ['census_tract', 'geoid', 'start_date', 'end_date']]

grouped_df2 = merged_df2.groupby(['ward_number', 'precinct_number'])[columns_to_average].mean()

# Round all numeric columns to 2 decimal places
grouped_df2 = grouped_df2.round(2)

# Merge back with the original df
final_df2 = pd.merge(df2, grouped_df2, 
                   on=['ward_number', 'precinct_number'],
                   how='left')




In [70]:
election_columns2 = [col for col in final_df2.columns if col not in ['voter_id_number', 'last_name', 'first_name', 'ward_number', 'precinct_number', 'longitude', 'latitude', 'census_tract', 'date_of_birth', 'date_of_registration'] + list(columns_to_average)]

melted_df2 = pd.melt(final_df2,
                   id_vars=['voter_id_number', 'last_name', 'first_name', 'ward_number', 'precinct_number', 'longitude', 'latitude', 'census_tract', 'date_of_birth', 'date_of_registration'] + list(columns_to_average),  
                   value_vars=election_columns2,  # These are your election date columns
                   var_name='election_date',  # This will contain the election dates
                   value_name='voted')  

# Sort the results for better readability
melted_df2 = melted_df2.sort_values(['voter_id_number', 'election_date'])

In [71]:
# Fill in blanks in voted column with 0
melted_df2['voted'] = melted_df2['voted'].fillna(0)

# Convert election_date and date_of_registration to datetime if they aren't already
melted_df2['election_date'] = pd.to_datetime(melted_df2['election_date'])
melted_df2['date_of_registration'] = pd.to_datetime(melted_df2['date_of_registration'])

# Create eligible column: 1 if registered at least 10 days before election, 0 otherwise
melted_df2['eligible'] = ((melted_df2['election_date'] - melted_df2['date_of_registration']).dt.days >= 10).astype(int)

# Sort the results for better readability
melted_df2 = melted_df2.sort_values(['voter_id_number', 'election_date'])

## Stack & Export

In [76]:
stacked_df = pd.concat([melted_df1, melted_df2], axis=0, ignore_index=True)
stacked_df

,voter_id_number,last_name,first_name,ward_number,precinct_number,longitude,latitude,census_tract,date_of_birth,date_of_registration,...,hispanic_cuban_percent,hispanic_other_any_percent,non_hispanic_total_percent,white_alone_percent,black_alone_percent,american_indian_alone_percent,native_hawaiian_alone_percent,election_date,voted,eligible
0,01AAA3148000,ARSENAULT,ANITA,2,1,-71.047913,42.389275,1603.0,1/31/1948,1988-05-15,...,0.3,20.69,68.87,49.62,9.10,0.04,0.0,2013-04-30,0,1
1,01AAA3148000,ARSENAULT,ANITA,2,1,-71.047913,42.389275,1603.0,1/31/1948,1988-05-15,...,0.3,20.69,68.87,49.62,9.10,0.04,0.0,2013-06-25,0,1
2,01AAA3148000,ARSENAULT,ANITA,2,1,-71.047913,42.389275,1603.0,1/31/1948,1988-05-15,...,0.3,20.69,68.87,49.62,9.10,0.04,0.0,2013-09-24,0,1
3,01AAA3148000,ARSENAULT,ANITA,2,1,-71.047913,42.389275,1603.0,1/31/1948,1988-05-15,...,0.3,20.69,68.87,49.62,9.10,0.04,0.0,2013-11-05,1,1
4,01AAA3148000,ARSENAULT,ANITA,2,1,-71.047913,42.389275,1603.0,1/31/1948,1988-05-15,...,0.3,20.69,68.87,49.62,9.10,0.04,0.0,2013-12-01,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249537,12ZNS0657000,ZEP,NICOLAS,2,3,-71.037586,42.391987,1604.0,12/6/1957,2023-07-03,...,0.0,59.37,32.22,20.29,6.37,0.00,0.0,2022-01-25,0,0
249538,12ZNS0657000,ZEP,NICOLAS,2,3,-71.037586,42.391987,1604.0,12/6/1957,2023-07-03,...,0.0,59.37,32.22,20.29,6.37,0.00,0.0,2022-09-06,0,0
249539,12ZNS0657000,ZEP,NICOLAS,2,3,-71.037586,42.391987,1604.0,12/6/1957,2023-07-03,...,0.0,59.37,32.22,20.29,6.37,0.00,0.0,2022-11-08,0,0
249540,12ZNS0657000,ZEP,NICOLAS,2,3,-71.037586,42.391987,1604.0,12/6/1957,2023-07-03,...,0.0,59.37,32.22,20.29,6.37,0.00,0.0,2023-09-26,0,1


In [70]:
melted_df.to_csv("chelsea_unpivoted_all.csv", index=False)
melted_df.to_excel('chelsea_unpivoted_all.xlsx', index=False)